# SEMTM0009 Week 4 Worksheet — computational template

**Controlling the disease.**

The plumbing is written. What is left to you is marked `# TODO`, and each
`# TODO` returns `np.nan` so the whole notebook runs at any stage and shows you
what is still missing.

**Units.** Q1 is entirely in days. In Q2, $\gamma$ is per day and $\mu$ is per
*year* — convert before adding.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "lines.linewidth": 1.8,
                     "axes.spines.top": False, "axes.spines.right": False})
np.set_printoptions(precision=4, suppress=True)

# ---------------- Question 1: days ----------------
R0_UNCONTROLLED = 5.0
GAMMA = 1/14          # recovery,   1/gamma = 14 d
D_I   = 1/5           # symptom onset, 1/d_I = 5 d
TAU   = 1/14          # release from quarantine, 1/tau = 14 d
BETA  = R0_UNCONTROLLED * GAMMA

---
# Q1(a) — quarantine changes $R_0$

In [ ]:
def R0_quarantine(R0, gamma=GAMMA, d_I=D_I):
    '''R0^Q: the reproduction number with quarantine (mu = 0).'''
    # TODO
    return np.nan

R0p = R0_quarantine(R0_UNCONTROLLED)
print(f"R0^Q = {R0p}")
print("eradicated by quarantine alone?" , (R0p < 1) if np.isfinite(R0p) else "?")

# Q1(b) — the capacity you need to guarantee no overflow

In [ ]:
def I_max(R0p):
    '''Peak prevalence of an SIR epidemic with reproduction number R0p.'''
    # TODO
    return np.nan


def Q_capacity(R0p, d_I=D_I, tau=TAU):
    '''The facility is fullest when dQ/dt = 0, i.e. tau*Q = d_I*I.'''
    # TODO
    return np.nan

print(f"I_max = {I_max(R0p)}")
print(f"Q_c   = {Q_capacity(R0p)}   ({100*Q_capacity(R0p):.2f}% of the population)")

# Q1(c) — is that bound tight? Simulate.

In [ ]:
def sirq(t, y, beta=BETA):
    S, I, Q, R = y
    return [-beta*S*I,
            beta*S*I - (GAMMA + D_I)*I,
            D_I*I - TAU*Q,
            GAMMA*I + TAU*Q]

s = solve_ivp(sirq, [0, 500], [1-1e-6, 1e-6, 0, 0],
              dense_output=True, rtol=1e-11, atol=1e-14)
t = np.linspace(0, 400, 20000)
S, I, Q, R = s.sol(t)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(t, I, color="tab:red", label="$I(t)$")
ax.plot(t, Q, color="tab:blue", label="$Q(t)$")
if np.isfinite(Q_capacity(R0p)):
    ax.axhline(Q_capacity(R0p), color="tab:orange", ls="--",
               label="analytic $Q_c$")
ax.set(xlabel="$t$ (days)", ylabel="fraction of population")
ax.legend(frameon=False); plt.show()

print(f"simulated peak I = {I.max():.6f} at t = {t[I.argmax()]:.1f} d")
print(f"simulated peak Q = {Q.max():.6f} at t = {t[Q.argmax()]:.1f} d")
print(f"lag between the two peaks = {t[Q.argmax()]-t[I.argmax()]:.1f} d")
if np.isfinite(Q_capacity(R0p)):
    print(f"the analytic bound is {100*(Q_capacity(R0p)/Q.max()-1):.1f}% above the truth")

**Question (c).** Is the bound conservative or optimistic, and why?

*Your answer:*

# Q1(d) — vaccinate to fit a 2% capacity

In [ ]:
CAPACITY = 0.02

def vaccination_for_capacity(Qc, R0p, d_I=D_I, tau=TAU):
    '''Coverage p that keeps the quarantine peak within capacity Qc.'''
    target = Qc * tau / d_I
    if not np.isfinite(I_max(R0p)):
        return np.nan, np.nan
    R0pp = brentq(lambda R: I_max(R) - target, 1.0000001, R0p)
    # TODO: convert R0^Q' into a coverage p
    p = np.nan
    return R0pp, p

R0pp, p = vaccination_for_capacity(CAPACITY, R0p)
print(f"target I_max = {CAPACITY*TAU/D_I:.6f}")
print(f"R0^Q' = {R0pp}   ->   p = {p}")

# Q1(e) — eradication thresholds

In [ ]:
def p_critical(R0):
    '''Prophylactic vaccination: R0 -> (1-p) R0, so eradication needs...'''
    # TODO
    return np.nan

print(f"with quarantine    (R0^Q={R0p}): p_c = {p_critical(R0p)}")
print(f"without quarantine (R0 =5)    : p_c = {p_critical(R0_UNCONTROLLED)}")

**Question (f).** (d) needs less vaccination than (e), yet (e) achieves strictly
more. What is (d) actually buying? Which would you recommend, and why?

*Your answer:*

---
# Question 2 — targeted vaccination

In [ ]:
B = np.array([[1.0, 0.1],
              [0.1, 0.2]])          # beta_ij: row = "to", column = "from"
GAMMA4 = 1/10                       # per day
MU4 = 1/(70*365)                    # per DAY -- the worksheet gives 1/70 per year
N = np.array([0.2, 0.8])            # n_H, n_L


def K_matrix(beta, n, pH=0.0, pL=0.0, gamma=GAMMA4, mu=MU4):
    '''Next-generation matrix with coverages pH, pL.

    S_i* = (1-p_i) n_i at the vaccinated disease-free equilibrium, and the
    group size weights the ROW -- the group being infected.
    '''
    # TODO: build K.  Hint: np.array([...])[:, None] broadcasts down rows.
    return np.nan * beta


def rho(K):
    '''Dominant eigenvalue.'''
    # TODO
    return np.nan

K0 = K_matrix(B, N)
print("K (no vaccination) =\n", K0)
print(f"R0 = rho(K) = {rho(K0)}")

# Q2(d) — random vaccination

In [ ]:
R0_struct = rho(K0)
p_random = 1 - 1/R0_struct if np.isfinite(R0_struct) else np.nan
print(f"random vaccination needs p_c = {p_random}")

# Q2(f) — the optimal targeted strategy

In [ ]:
def pH_min(beta=B, n=N, gamma=GAMMA4, mu=MU4):
    '''Below this, the high-risk group sustains the epidemic alone and no pL helps.

    Set pL = 1; K becomes upper-triangular with a single non-zero row, so its
    dominant eigenvalue is the top-left entry.  Solve that = 1.
    '''
    # TODO
    return np.nan


def pL_threshold(pH):
    '''The pL that makes rho(K) = 1, or nan if no pL can.'''
    if not np.isfinite(rho(K_matrix(B, N, pH, 1.0))):
        return np.nan
    if rho(K_matrix(B, N, pH, 1.0)) > 1:
        return np.nan            # even pL = 1 is not enough
    if rho(K_matrix(B, N, pH, 0.0)) < 1:
        return 0.0               # no low-risk vaccination needed
    return brentq(lambda x: rho(K_matrix(B, N, pH, x)) - 1, 0, 1, xtol=1e-14)

print(f"pH_min = {pH_min()}")

In [ ]:
if np.isfinite(pH_min()):
    pHs = np.linspace(pH_min(), 1, 2000)
    pLs = np.array([pL_threshold(x) for x in pHs])
    total = N[0]*pHs + N[1]*pLs
    i = np.nanargmin(total)
    print(f"optimum: pH = {pHs[i]:.4f}, pL = {pLs[i]:.4f}, "
          f"total = {100*total[i]:.2f}%")
    print(f"random vaccination        : {100*p_random:.2f}%")
    print(f"saving                    : {100*(p_random-total[i]):.2f} percentage points")

    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.8))
    a.plot(pHs, pLs, color="tab:blue"); a.plot(pHs[i], pLs[i], "k*", ms=14)
    a.set(xlabel="$p_H$", ylabel="$p_L$ for $\\rho(K)=1$")
    a.set_title("eradication boundary")
    b.plot(pHs, 100*total, color="tab:red")
    b.axhline(100*p_random, color="grey", ls="--", label="random")
    b.plot(pHs[i], 100*total[i], "k*", ms=14)
    b.set(xlabel="$p_H$", ylabel="total vaccinated (%)", ylim=(46.9, 57))
    b.legend(frameon=False); b.set_title("total doses")
    plt.tight_layout(); plt.show()
else:
    print("fill in K_matrix, rho and pH_min first")

# Q2(g) — how well determined is the optimum?

In [ ]:
if np.isfinite(pH_min()):
    lo, hi = 0.72, 0.75
    m = (pHs >= lo) & (pHs <= hi)
    spread = 100*(total[m].max() - total[m].min())
    print(f"over pH in [{lo}, {hi}] the total varies by {spread:.4f} percentage points")

    # how far can pH move before the total rises by 0.1 percentage points?
    band = pHs[100*total <= 100*total[i] + 0.1]
    print(f"\npH values within 0.1 pp of the optimum: "
          f"[{band.min():.3f}, {band.max():.3f}]  (width {band.max()-band.min():.3f})")

**Question (g).** To how many decimals should $p_H$ be reported? Is a flat
optimum good news or bad news — answer separately for a policymaker and for
someone reporting the argmin.

*Your answer:*

# Q2(h) (optional) — which mixing assumption is this?

In [ ]:
print(f"det(beta) = {np.linalg.det(B):.4f}")
print(f"rank(beta) = {np.linalg.matrix_rank(B)}")

**Question (h).** Proportional or assortative? The lecture says that under
*proportional* mixing you should vaccinate the highest-contact group entirely
before anyone else — but here the optimum has $p_H<1$. Reconcile these.

*Your answer:*